In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import gc
import math

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms.functional as VF
from torch.autograd.graph import saved_tensors_hooks

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from torch_utils import gpu_utils
from torch_utils import cuda_utils
from torch_utils import training_utils

In [2]:
device = 'cuda'
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.8.0+cu128


## Dataset

In [3]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./DATA",
    train=True,
    download=True,
    transform=transform)

valid_dataset = datasets.MNIST(
    root="./DATA",
    train=False,
    download=True,
    transform=transforms.ToTensor())    

## Model class

In [4]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 128)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        return x

In [5]:
class TorchTrainer:
    def __init__(self, model, optimizer, criterion, device, callbacks=None):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.device = device
        self.callbacks = callbacks or []
        self.model.to(self.device)

        self._reset_training()

    def _training_step(self, batch_x, batch_y):
        logs = {
            "step_idx": self.step_idx
        }
                
        for cb in self.callbacks:
            cb.on_step_start(trainer = self, logs = logs)        
        self.model.train()
        batch_x = batch_x.to(self.device)
        batch_y = batch_y.to(self.device)
        self.optimizer.zero_grad()
        output = self.model(batch_x)
        loss = self.criterion(output, batch_y)
        loss.backward()
        self.optimizer.step()
        self.train_loss_per_step.append(loss.item())

        logs = {
            "step_idx": self.step_idx,
            "train_loss_step": loss.item(),
            "num_samples_step": len(batch_x)
        }
        
        for cb in self.callbacks:
            cb.on_step_end(trainer = self, logs = logs)

        self.step_idx += 1
        return loss.item()

    def evaluate_epoch(self, loader):
        self.model.eval()
        with torch.no_grad():
            running_loss = 0.0
            num_samples = 0
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                output = self.model(batch_x)
                loss = self.criterion(output, batch_y)
                running_loss += loss.item() * len(batch_x)
                num_samples += len(batch_x)
            running_loss = running_loss / num_samples
            return running_loss

    def _reset_training(self):
        self.step_idx = 0
        self.epoch_idx = 0
        self.train_loss_per_epoch = []
        self.train_loss_per_step = []
    
    def fit(self, train_loader, n_epochs):
        self._reset_training()
        
        for epoch in range(n_epochs):
            logs = {
                "epoch_idx": self.epoch_idx
            }
            for cb in self.callbacks:
                cb.on_epoch_start(trainer = self, logs = logs)            
            loss_epoch = 0
            num_obs_epoch = 0
            for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
                loss_step = self._training_step(batch_x, batch_y)
                loss_epoch +=  loss_step * len(batch_x)
                num_obs_epoch += len(batch_x)
            loss_epoch = loss_epoch / num_obs_epoch
            self.train_loss_per_epoch.append(loss_epoch)

            logs = {
                "epoch_idx": self.epoch_idx,
                "train_loss_epoch": loss_epoch,
                "num_samples_epoch": num_obs_epoch
            }
            for cb in self.callbacks:
                cb.on_epoch_end(trainer = self, logs = logs)

            self.epoch_idx += 1

    def named_parameters(self, required_grad = True, include_bias = False, include_empty_grad=False):
        for name, param in self.model.named_parameters():
            if required_grad and not param.requires_grad:
                continue
            if not include_bias and name.endswith(".bias"):
                continue
            if required_grad and not include_empty_grad and param.grad is None:
                continue
            yield name, param

        

In [6]:
class TrainerCallback:
    def on_epoch_start(self, trainer, logs): pass
    def on_epoch_end(self, trainer, logs): pass
    def on_step_start(self, trainer, logs): pass
    def on_step_end(self, trainer, logs): pass

class TrainerLogger(TrainerCallback):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None):
        self.tb_writer = tb_writer
        self.group_name_step = group_name_step
        self.group_name_epoch = group_name_epoch

    def _log(self, trainer, group_name, step, logs, **kwargs):
        raise NotImplementedError(f"{self.__class__.__name__} must implement _log()")
        pass
        
    def on_step_end(self, trainer, logs):
        if self.group_name_step is not None:
            self._log(
                trainer = trainer,
                group_name = self.group_name_step,
                step = logs["step_idx"],
                logs = logs)

    def on_epoch_end(self, trainer, logs):
        if self.group_name_epoch is not None:
            self._log(
                trainer = trainer,
                group_name = self.group_name_epoch,
                step = logs["epoch_idx"],
                logs = logs)        

class ValidationLossLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_epoch, valid_loader):
        super().__init__(tb_writer = tb_writer, group_name_step = None, group_name_epoch = group_name_epoch)
        self.valid_loader = valid_loader

    def _log(self, trainer, group_name, step, logs, **kwargs):
        valid_loss = trainer.evaluate_epoch(loader = self.valid_loader)
        self.tb_writer.add_scalar(group_name, valid_loss, step)

class ParamNormLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None, is_gradient = False, is_scaled = True):
        super().__init__(tb_writer = tb_writer, group_name_step = group_name_step, group_name_epoch = group_name_epoch)
        self.is_gradient = is_gradient
        self.is_scaled = is_scaled

    def _log(self, trainer, group_name, step, logs, **kwargs):
        norm_total = 0
        numel_total = 0
        for name, param in trainer.named_parameters(required_grad=True, include_bias=False, include_empty_grad=not self.is_gradient):
            t = param.grad.detach() if self.is_gradient else param.detach()
            norm = t.norm(2)
            norm_total += norm ** 2
            if self.is_scaled:
                norm = norm / (t.numel() ** 0.5)
                numel_total += t.numel()
            self.tb_writer.add_scalar(f"{group_name}/{name}", norm, step)
        norm_total = norm_total ** 0.5
        if self.is_scaled:
            norm_total = norm_total / (numel_total ** 0.5)
        self.tb_writer.add_scalar(f"{group_name}/total", norm_total, step) 


class ParamDistributionLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None, is_gradient = False):
        super().__init__(tb_writer = tb_writer, group_name_step = group_name_step, group_name_epoch = group_name_epoch)
        self.is_gradient = is_gradient

    def _log(self, trainer, group_name, step, logs, **kwargs):
        for name, param in trainer.named_parameters(required_grad=True, include_bias=False, include_empty_grad=not self.is_gradient):
            t = (param.grad if self.is_gradient else param).detach().cpu()
            self.tb_writer.add_histogram(f"{group_name}/{name}", t, step)
        

class AdamUpdateRatioLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None):
        super().__init__(tb_writer = tb_writer, group_name_step = group_name_step, group_name_epoch = group_name_epoch)

    def _log(self, trainer, group_name, step, logs, **kwargs):
        model = trainer.model
        optimizer = trainer.optimizer
        beta1, beta2 = optimizer.param_groups[0]["betas"]
        for name, param in trainer.named_parameters(required_grad=True, include_bias=False, include_empty_grad=False):
            num_of_updates = optimizer.state[param]['step'] 
            m_t = optimizer.state[param]['exp_avg'] / (1 - beta1 ** num_of_updates)
            v_t = optimizer.state[param]['exp_avg_sq'] / (1 - beta2 ** num_of_updates)
            base_lr = optimizer.param_groups[0]["lr"]
            eps = optimizer.param_groups[0]["eps"]
            effective_update = base_lr * m_t / (v_t ** 0.5 + eps)
            update_ratio = effective_update.abs().mean() / param.detach().abs().mean()
            self.tb_writer.add_scalar(f"{group_name}/{name}", update_ratio, step)                

class TrainLossLogger(TrainerLogger):        
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None):
        super().__init__(
            tb_writer = tb_writer,
            group_name_step= group_name_step,
            group_name_epoch = group_name_epoch)

    def _log(self, trainer, group_name, step, logs, **kwargs):
        self.tb_writer.add_scalar(group_name, logs[kwargs['loss_key']], step)

    def on_step_end(self, trainer, logs):
        if self.group_name_step is not None:
            self._log(
                trainer = trainer,
                group_name = self.group_name_step,
                step = logs["step_idx"],
                logs = logs,
                loss_key = "train_loss_step")

    def on_epoch_end(self, trainer, logs):
        if self.group_name_epoch is not None:
            self._log(
                trainer = trainer,
                group_name = self.group_name_epoch,
                step = logs["epoch_idx"],
                logs = logs,
                loss_key = "train_loss_epoch")    

class PerformanceLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None):
        super().__init__(tb_writer = tb_writer, group_name_step = group_name_step, group_name_epoch = group_name_epoch)

    def _log(self, trainer, group_name, step, logs, **kwargs):
        
        duration = time.time() - kwargs['start_time']
        self.tb_writer.add_scalar(f"{group_name}/duration", duration, step)

        throughput = kwargs["num_samples"] / duration
        self.tb_writer.add_scalar(f"{group_name}/troughput", throughput, step)
        
    def on_epoch_start(self, trainer, logs):
        self.epoch_start_time = time.time()
        
    def on_step_start(self, trainer, logs):
        torch.cuda.synchronize(trainer.device)
        self.step_start_time = time.time()

    def on_step_end(self, trainer, logs):
        if self.group_name_step is not None:
            torch.cuda.synchronize(trainer.device)
            self._log(
                trainer = trainer,
                group_name = self.group_name_step,
                step = logs["step_idx"],
                logs = logs,
                start_time = self.step_start_time,
                num_samples = logs["num_samples_step"])

    def on_epoch_end(self, trainer, logs):
        if self.group_name_epoch is not None:
            self._log(
                trainer = trainer,
                group_name = self.group_name_epoch,
                step = logs["epoch_idx"],
                logs = logs,
                start_time = self.epoch_start_time,
                num_samples = logs["num_samples_epoch"])

class LearningRateLogger(TrainerLogger):
    def __init__(self, tb_writer, group_name_step = None, group_name_epoch = None):
        super().__init__(tb_writer = tb_writer, group_name_step = group_name_step, group_name_epoch = group_name_epoch)

    def _log(self, trainer, group_name, step, logs, **kwargs):
        current_lr = trainer.optimizer.param_groups[0]["lr"]
        self.tb_writer.add_scalar(group_name, current_lr, step)

In [7]:
model = SimpleNN().to(device)
optimizer = optim.AdamW(
    params=model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
criterion = nn.CrossEntropyLoss()
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    #persistent_workers=True,
    drop_last=True)

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
   # persistent_workers=True,
    drop_last=False)

In [8]:
exp_folder = f'runs/model_{int(time.time())}'
writer_train = SummaryWriter(f"{exp_folder}/train")
writer_valid = SummaryWriter(f"{exp_folder}/valid")

callbacks = [
    TrainLossLogger(tb_writer = writer_train, 
                    group_name_step = "Loss/step", 
                    group_name_epoch = "Loss/epoch"),
    
    ParamNormLogger(tb_writer = writer_train, 
                    is_gradient = False, 
                    is_scaled = True, 
                    group_name_epoch = "ParamNormEpoch"),
    
    ParamNormLogger(tb_writer = writer_train, 
                    is_gradient = True, 
                    is_scaled = True, 
                    group_name_epoch = "GradNormEpoch"),
    
    ParamDistributionLogger(tb_writer = writer_train, 
                    is_gradient = False, 
                    group_name_epoch = "ParamDistribution"),

    ParamDistributionLogger(tb_writer = writer_train, 
                    is_gradient = True, 
                    group_name_epoch = "GradDistribution"),   

    ValidationLossLogger(tb_writer = writer_valid, 
                             group_name_epoch = "Loss/epoch", 
                             valid_loader = valid_loader),    

    AdamUpdateRatioLogger(tb_writer = writer_train, 
                          group_name_step = None, 
                          group_name_epoch = "UpdateRatioEpoch"),
    
    PerformanceLogger(tb_writer = writer_train, 
                    group_name_step = "StepPerformance",
                    group_name_epoch = "EpochPerformance"),

    LearningRateLogger(tb_writer = writer_train,
                       group_name_step = "LearningRateStep")
]

trainer = TorchTrainer(
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    callbacks = callbacks)

trainer.fit(
    train_loader = train_loader,
    n_epochs = 20)

In [9]:
layout = {
    "Losses": {
        # This will show one line per RUN (train + valid) for tag "Loss/epoch"
        "Train_vs_Valid_epoch_loss": ["Multiline", ["Loss/epoch"]],
        # Only train has step loss, so only that run will show here
        "Train_step_loss": ["Scalar", "Loss/step"],
    },
    "Gradients": {
        "GradNorm_per_layer": [
            "Multiline",
            [
                "GradNormEpoch/fc1.weight",
                "GradNormEpoch/fc2.weight",
                "GradNormEpoch/fc3.weight",
            ],
        ],
    },
    "Optimization": {
        "LearningRate": ["Scalar", "LearningRateStep"],
    },
}
writer_train.add_custom_scalars(layout)
